# Análise exploratória e pré-processamento — *dataset* Titanic

**Integrantes:** 
 
Diogo Galrão Carvalho.

Felipe Artur Macedo Lima.  

Luan Cavalcante Dias Rodrigues.


**Objetivo:** conhecer a estrutura do *dataset* Titanic, classificar as variáveis, identificar valores ausentes e inconsistências e preparar os dados para as análises seguintes.

**Fonte:** Kaggle — [`yasserh/titanic-dataset`](https://www.kaggle.com/datasets/yasserh/titanic-dataset), 891 passageiros e 12 colunas.

## Carregamento dos dados

O *dataset* é baixado com `kagglehub`. O token do Kaggle **não fica gravado no notebook**: ele é lido de uma variável de ambiente ou digitado na hora.

In [1]:
import os
import kagglehub

if "KAGGLE_API_TOKEN" not in os.environ:
    from getpass import getpass
    os.environ["KAGGLE_API_TOKEN"] = getpass("Cole seu token do Kaggle: ")

os.environ.pop("KAGGLE_USERNAME", None)
os.environ.pop("KAGGLE_KEY", None)

path = kagglehub.dataset_download("yasserh/titanic-dataset")
print("Path to dataset file:", path)

Path to dataset file: /home/luanzra/.cache/kagglehub/datasets/yasserh/titanic-dataset/versions/1


In [2]:
os.listdir(path)

['Titanic-Dataset.csv']

In [3]:
import pandas as pd

df = pd.read_csv(os.path.join(path, "Titanic-Dataset.csv"))

## Bloco A — Conhecendo o dataset

### Passo 1 — Primeira olhada: `.head()`, `.info()` e `.describe()`

Antes de qualquer análise, três perguntas: **como os dados chegam** (`head`), **qual o tipo de cada coluna e quantos valores existem** (`info`) e **qual o tamanho do dataset** (`shape`). O `.describe()` aparece no Passo 3, aplicado só às variáveis quantitativas.

In [4]:
# Primeiras linhas: como os dados chegam "crus"
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Estrutura: tipo de cada coluna, quantos não nulos e uso de memória
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
# Tamanho do dataset
linhas, colunas = df.shape
print(f"Tamanho do dataset:\nLinhas: {linhas}\nColunas: {colunas}")

Tamanho do dataset:
Linhas: 891
Colunas: 12


#### Tabela-resumo por coluna

Junta numa só tabela o que o `.info()` e outros comandos mostrariam separados: tipo, quantidade e percentual de nulos, número de valores únicos e um exemplo de valor. Ordenada pelo percentual de nulos, para os problemas aparecerem primeiro.

In [7]:
resumo = pd.DataFrame({
    "tipo": df.dtypes,
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(1),
    "valores únicos": df.nunique(),
    "exemplo": df.loc[0],
})

resumo.sort_values("% nulos", ascending=False)

,tipo,nulos,% nulos,valores únicos,exemplo
Cabin,str,687,77.1,147,NaN
Age,float64,177,19.9,88,22.0
Embarked,str,2,0.2,3,S
PassengerId,int64,0,0.0,891,1
Name,str,0,0.0,891,"Braund, Mr. Owen Harris"
Pclass,int64,0,0.0,3,3
Survived,int64,0,0.0,2,0
Sex,str,0,0.0,2,male
Parch,int64,0,0.0,7,0
SibSp,int64,0,0.0,7,1


**O que a tabela mostra:**
- **Valores ausentes** em apenas três colunas: `Cabin` (77,1%), `Age` (19,9%) e `Embarked` (2 linhas, 0,2%). O tratamento de cada uma será decidido no Passo 7.
- **Valores únicos** ajudam a classificar: `PassengerId` e `Name` têm 891 valores (um por pessoa, ou seja, são identificadores), e `Ticket` e `Cabin` têm muitos valores diferentes (alta cardinalidade).
- O **tipo do pandas** (`int64`, `float64`, `str`) não basta para classificar. `Survived` e `Pclass` são `int64`, mas os números são rótulos. A classificação correta vem no Passo 2.

### Passo 2 — Classificação das variáveis

**Critério usado:** uma variável é **quantitativa** quando o número mede ou conta algo, ou seja, somar e tirar média produz um resultado com sentido. É **qualitativa** quando o valor é um rótulo, mesmo que esteja escrito como número.

As listas abaixo guardam a classificação no código, e todas as células seguintes usam essas listas. A conferência no final garante que nenhuma das 12 colunas ficou de fora.

In [8]:
# Separar colunas por papel e por subtipo
alvo = "Survived"                                  # qualitativa nominal binária

# Fora da análise direta (rótulos únicos ou quase únicos)
ids = ["PassengerId", "Name", "Ticket", "Cabin"]

# Qualitativas
qualitativas_nominais = ["Sex", "Embarked"]
qualitativas_ordinais = ["Pclass"]

# Quantitativas
quantitativas_discretas = ["SibSp", "Parch"]
quantitativas_continuas = ["Age", "Fare"]

# Atalhos usados nas próximas células
categoricas = qualitativas_nominais + qualitativas_ordinais
numericas = quantitativas_continuas + quantitativas_discretas

# Conferência: nenhuma coluna pode ficar de fora nem aparecer duas vezes
classificadas = [alvo] + ids + categoricas + numericas
assert sorted(classificadas) == sorted(df.columns), set(df.columns) ^ set(classificadas)

print(f"Alvo: {alvo}")
print(f"Identificadores (fora da análise direta): {ids}")
print(f"Qualitativas nominais: {qualitativas_nominais}")
print(f"Qualitativas ordinais: {qualitativas_ordinais}")
print(f"Quantitativas discretas: {quantitativas_discretas}")
print(f"Quantitativas contínuas: {quantitativas_continuas}")
print(f"Total classificado: {len(classificadas)} de {df.shape[1]} colunas")

Alvo: Survived
Identificadores (fora da análise direta): ['PassengerId', 'Name', 'Ticket', 'Cabin']
Qualitativas nominais: ['Sex', 'Embarked']
Qualitativas ordinais: ['Pclass']
Quantitativas discretas: ['SibSp', 'Parch']
Quantitativas contínuas: ['Age', 'Fare']
Total classificado: 12 de 12 colunas


#### Tabela de classificação

**Critério usado:** uma variável é **quantitativa** quando o número mede ou conta algo, ou seja, somar e tirar média produz um resultado com sentido. É **qualitativa** quando o valor é um rótulo, mesmo que esteja escrito como número.

| Coluna | Tipo | Subtipo | Justificativa | Papel na análise |
|---|---|---|---|---|
| `Survived` | Qualitativa | Nominal (binária) | 0/1 é um código para "não sobreviveu/sobreviveu". Não existe "meia sobrevivência". A média (0,38) só tem leitura porque, numa variável 0/1, ela vira a **proporção** de sobreviventes. | **Variável alvo** |
| `Pclass` | Qualitativa | **Ordinal** | Há ordem (1ª > 2ª > 3ª classe), mas a distância entre as classes não é medida. A 1ª não vale "três vezes" a 3ª, e a média 2,3 não significa nada. | Explicativa (indicador socioeconômico) |
| `Sex` | Qualitativa | Nominal (binária) | Duas categorias sem ordem. | Explicativa |
| `Embarked` | Qualitativa | Nominal | Porto de embarque (C = Cherbourg, Q = Queenstown, S = Southampton), sem ordem. 2 ausentes. | Explicativa |
| `Age` | Quantitativa | Contínua | Idade em anos, uma medida que admite frações (0,42 = cerca de 5 meses). 19,9% ausentes. | Explicativa |
| `Fare` | Quantitativa | Contínua | Valor monetário (libras) com casas decimais. **Atenção:** é o preço do bilhete, não por pessoa (ver inconsistências). | Explicativa |
| `SibSp` | Quantitativa | **Discreta** | **Contagem** de irmãos e cônjuges a bordo (0, 1, 2…). Só aceita inteiros, e a soma e a média fazem sentido. | Explicativa (tamanho da família) |
| `Parch` | Quantitativa | **Discreta** | **Contagem** de pais e filhos a bordo. Mesmo raciocínio de `SibSp`. | Explicativa (tamanho da família) |
| `PassengerId` | Qualitativa | Nominal (identificador) | Número sequencial que só identifica a linha (891 valores únicos). Somar ids não faz sentido. | Descartar |
| `Name` | Qualitativa | Nominal (texto livre) | Único por pessoa. | Descartar (o título, como Mr/Mrs/Master, pode ser extraído depois) |
| `Ticket` | Qualitativa | Nominal (alta cardinalidade) | 681 códigos para 891 pessoas, com prefixos irregulares. | Descartar como variável, mas serve para contar **quantas pessoas dividem o bilhete** |
| `Cabin` | Qualitativa | Nominal (alta cardinalidade) | 147 códigos e **77,1% ausentes**. Algumas linhas listam várias cabines ("C23 C25 C27"). | Descartar ou derivar o **deck** (letra) e um indicador "tem cabine registrada" |

**Por que separar assim importa para o resto da análise:**
- **Qualitativas** (nominais e ordinais) são resumidas por **contagem e proporção** e vão para gráficos de barras. Na codificação (Passo 9), as **nominais** recebem *one-hot*, porque não têm ordem. A **ordinal** `Pclass` pode manter os códigos 1, 2 e 3, já que a ordem é real.
- **Quantitativas** são resumidas por média, mediana, desvio padrão e quartis e vão para histogramas e boxplots. A análise de *outliers* (Passo 8) e a normalização (Passo 10) só se aplicam a elas, principalmente às **contínuas**.
- `SibSp` e `Parch` são **quantitativas discretas**: entram no `describe()` e na análise de outliers. Como têm poucos valores distintos, também podem ser **exibidas** como barras, mas isso é uma escolha de gráfico, não muda a classificação.
- `Survived` está em 0/1 por conveniência de código. Ela é tratada como categoria (a taxa de sobrevivência é uma proporção) e nunca é normalizada.

### Passo 3 — Estatísticas descritivas das variáveis quantitativas

O `.describe()` é aplicado **só às quantitativas** (contínuas e discretas). Para as qualitativas, média e desvio padrão não têm sentido, e o resumo adequado é a contagem de cada categoria, feita logo depois. Na tabela, a coluna **50% é a mediana**.

In [9]:
df[numericas].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,714.0,29.70,14.53,0.42,20.12,28.00,38.0,80.00
Fare,891.0,32.20,49.69,0.00,7.91,14.45,31.0,512.33
SibSp,891.0,0.52,1.10,0.00,0.00,0.00,1.0,8.00
Parch,891.0,0.38,0.81,0.00,0.00,0.00,0.0,6.00


**Leitura:**
- **`Age`:** média (29,7) e mediana (28) próximas, o que indica distribuição **pouco assimétrica**. Vai de bebês (0,42) a 80 anos, e metade dos passageiros tem entre 20 e 38 anos.
- **`Fare`:** média (32,2) é mais que o **dobro da mediana** (14,45), com desvio padrão (49,7) maior que a própria média e máximo de 512. Esses números indicam **forte assimetria à direita**: a maioria pagou pouco, e poucos valores muito altos puxam a média. É a principal candidata a *outliers* (Passo 8).
- **`SibSp` e `Parch`:** a mediana é 0 nas duas, então **a maioria viajou sem irmãos/cônjuge e sem pais/filhos**. Os máximos (8 e 6) são famílias grandes, raras mas reais.

#### Frequência das categorias e checagens de qualidade

Para as qualitativas, o resumo é a **contagem por categoria**. `SibSp` e `Parch` entram aqui também porque têm poucos valores distintos.

In [10]:
for col in categoricas + quantitativas_discretas:
    print(f"\n---{col}---")
    print(df[col].value_counts(dropna=False).sort_index())


---Sex---
Sex
female    314
male      577
Name: count, dtype: int64

---Embarked---
Embarked
C      168
Q       77
S      644
NaN      2
Name: count, dtype: int64

---Pclass---
Pclass
1    216
2    184
3    491
Name: count, dtype: int64

---SibSp---
SibSp
0    608
1    209
2     28
3     16
4     18
5      5
8      7
Name: count, dtype: int64

---Parch---
Parch
0    678
1    118
2     80
3      5
4      4
5      5
6      1
Name: count, dtype: int64


In [11]:
# Checagens de qualidade
print(f"Linhas duplicadas: {df.duplicated().sum()}")
print(f"Taxa de sobrevivência: {df[alvo].mean():.1%}")

Linhas duplicadas: 0
Taxa de sobrevivência: 38.4%


**Leitura:**
- As categorias são **desiguais**: 55% dos passageiros estavam na 3ª classe (491 de 891), 65% eram homens (577) e 72% embarcaram em Southampton (644). Grupos pequenos, como Queenstown (77), terão taxas de sobrevivência menos estáveis nas comparações.
- **Nenhuma linha duplicada**, então não é preciso remover registros repetidos.
- **Taxa de sobrevivência de 38,4%:** a variável alvo é **desbalanceada** (cerca de 6 mortos para cada 4 sobreviventes). Isso será analisado nos Passos 4 e 6.

### Passo 2 (cont.) — Inconsistências iniciais

Três valores nos próprios resultados acima pedem verificação antes de seguir: `Fare` mínima = 0, `Ticket` com menos valores únicos que passageiros e `Age` mínima = 0,42.

#### 1) Tarifa zero: faz sentido pagar 0 por uma passagem?

In [12]:
tarifa_zero = df[df["Fare"] == 0]
print(f"Passageiros com Fare = 0: {len(tarifa_zero)} ({len(tarifa_zero)/len(df):.1%})")
print(tarifa_zero["Sex"].value_counts().to_dict(), "| Embarque:", tarifa_zero["Embarked"].value_counts().to_dict())
print(f"Sobreviveram: {tarifa_zero['Survived'].sum()} de {len(tarifa_zero)}")
tarifa_zero[["Name", "Pclass", "Ticket", "Age", "Survived"]].sort_values("Ticket")

Passageiros com Fare = 0: 15 (1.7%)
{'male': 15} | Embarque: {'S': 15}
Sobreviveram: 1 de 15


,Name,Pclass,Ticket,Age,Survived
806,"Andrews, Mr. Thomas Jr",1,112050,39.0,0
633,"Parr, Mr. William Henry Marsh",1,112052,NaN,0
815,"Fry, Mr. Richard",1,112058,NaN,0
263,"Harrison, Mr. William",1,112059,40.0,0
822,"Reuchlin, Jonkheer. John George",1,19972,38.0,0
277,"Parkes, Mr. Francis ""Frank""",2,239853,NaN,0
413,"Cunningham, Mr. Alfred Fleming",2,239853,NaN,0
466,"Campbell, Mr. William",2,239853,NaN,0
481,"Frost, Mr. Anthony Wood ""Archie""",2,239854,NaN,0
732,"Knight, Mr. Robert J",2,239855,NaN,0


#### 2) Bilhetes compartilhados: `Fare` é o preço por pessoa ou por bilhete?

`Ticket` tem 681 códigos para 891 pessoas, ou seja, há bilhetes usados por mais de uma pessoa. Se a tarifa se repete igual para todos do mesmo bilhete, ela é o preço do **bilhete inteiro**.

In [13]:
# Quantas pessoas usam o mesmo bilhete (valor repetido em cada linha)
pessoas_por_ticket = df.groupby("Ticket")["Ticket"].transform("size")
compartilhados = df.loc[pessoas_por_ticket > 1, "Ticket"].nunique()
print(f"Bilhetes usados por mais de uma pessoa: {compartilhados}")
print(f"Passageiros nesses bilhetes: {(pessoas_por_ticket > 1).sum()} ({(pessoas_por_ticket > 1).mean():.1%})")

# A Fare se repete dentro do mesmo bilhete?
fares_por_ticket = df.groupby("Ticket")["Fare"].nunique()
print(f"Bilhetes compartilhados com Fare diferente entre as pessoas: {(fares_por_ticket > 1).sum()}")

# Exemplos: a maior tarifa do dataset e a família Sage
print("\nMaior tarifa do dataset:")
display(df.loc[df["Fare"] == df["Fare"].max(), ["Name", "Ticket", "Pclass", "Fare"]])
print("Família Sage (mesmo bilhete):")
display(df.loc[df["Ticket"] == "CA. 2343", ["Name", "Ticket", "Pclass", "Fare"]])

Bilhetes usados por mais de uma pessoa: 134
Passageiros nesses bilhetes: 344 (38.6%)
Bilhetes compartilhados com Fare diferente entre as pessoas: 1

Maior tarifa do dataset:


,Name,Ticket,Pclass,Fare
258,"Ward, Miss. Anna",PC 17755,1,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",PC 17755,1,512.3292
737,"Lesurer, Mr. Gustave J",PC 17755,1,512.3292


Família Sage (mesmo bilhete):


,Name,Ticket,Pclass,Fare
159,"Sage, Master. Thomas Henry",CA. 2343,3,69.55
180,"Sage, Miss. Constance Gladys",CA. 2343,3,69.55
201,"Sage, Mr. Frederick",CA. 2343,3,69.55
324,"Sage, Mr. George John Jr",CA. 2343,3,69.55
792,"Sage, Miss. Stella Anna",CA. 2343,3,69.55
846,"Sage, Mr. Douglas Bullen",CA. 2343,3,69.55
863,"Sage, Miss. Dorothy Edith ""Dolly""",CA. 2343,3,69.55


In [14]:
# Quanto muda se dividirmos a tarifa pelo número de pessoas no bilhete?
comparacao = pd.DataFrame({
    "Fare (por bilhete)": df["Fare"],
    "Fare por pessoa (estimada)": df["Fare"] / pessoas_por_ticket,
}).describe().T.round(2)

comparacao

,count,mean,std,min,25%,50%,75%,max
Fare (por bilhete),891.0,32.20,49.69,0.0,7.91,14.45,31.00,512.33
Fare por pessoa (estimada),891.0,17.79,21.22,0.0,7.76,8.85,24.29,221.78


#### 3) Idades fracionárias: erro de digitação ou dado real?

In [15]:
bebes = df[df["Age"] < 1]
print(f"Passageiros com menos de 1 ano: {len(bebes)} | sobreviveram: {bebes['Survived'].sum()}")
display(bebes[["Name", "Age", "Pclass", "Survived"]])

# Idades acima de 1 ano terminadas em .5 (no dicionário do dataset, indicam idade ESTIMADA)
estimadas = df[(df["Age"] > 1) & (df["Age"] % 1 == 0.5)]
print(f"Idades estimadas (xx.5): {len(estimadas)}")

Passageiros com menos de 1 ano: 7 | sobreviveram: 7


,Name,Age,Pclass,Survived
78,"Caldwell, Master. Alden Gates",0.83,2,1
305,"Allison, Master. Hudson Trevor",0.92,1,1
469,"Baclini, Miss. Helene Barbara",0.75,3,1
644,"Baclini, Miss. Eugenie",0.75,3,1
755,"Hamalainen, Master. Viljo",0.67,2,1
803,"Thomas, Master. Assad Alexander",0.42,3,1
831,"Richards, Master. George Sibley",0.83,2,1


Idades estimadas (xx.5): 18


#### Leitura das inconsistências

1. **`Fare` = 0 (15 passageiros, 1,7%).** Em princípio ninguém paga zero por uma passagem, então o valor merece checagem. O padrão, porém, não é aleatório: são **todos homens, todos embarcados em Southampton**, distribuídos pelas três classes, e vários têm bilhetes em sequência (`LINE`, `2398xx`, `1120xx`). Isso indica **passagens de cortesia** (funcionários ou convidados das empresas envolvidas, por exemplo Thomas Andrews, projetista do navio), não erro de digitação. **Decisão:** manter as linhas e registrar o motivo. Não serão tratadas como ausentes nem removidas como outliers, mas vale lembrar delas ao interpretar `Fare` (puxam o mínimo e a média para baixo).

2. **`Ticket` com 681 valores para 891 pessoas.** São **134 bilhetes compartilhados por 344 passageiros (38,6%)**, e em 133 deles a `Fare` é **idêntica para todos do grupo**. Conclusão: **`Fare` é o preço do bilhete inteiro, não por pessoa**. Exemplos: a família Sage (7 pessoas, 3ª classe) aparece com 69,55 cada, e o máximo de 512,33 é **um único bilhete para 3 pessoas**. **Impacto na análise de outliers (Passo 8):** parte dos valores altos de `Fare` não representa pessoas que pagaram caro, e sim **grupos**. Dividindo pelo tamanho do grupo, a média cai de 32,2 para cerca de 17,8 e o máximo de 512 para cerca de 222. *Limitação:* o dataset tem só 891 dos passageiros do navio, então o tamanho do grupo pode estar subestimado. Uma `Fare` por pessoa é candidata a variável derivada, e a decisão fica para o Passo 8.

3. **`Age` mínima = 0,42.** Não é erro: são **7 bebês com menos de 1 ano** (0,42 ano ≈ 5 meses), todos com título *Master/Miss*, coerentes com crianças, e **todos sobreviveram**. A idade fracionária representa meses. Há ainda **18 idades terminadas em .5** acima de 1 ano, que segundo a documentação do dataset são **idades estimadas**. Isso não invalida o dado, mas mostra que parte de `Age` é aproximada, além dos 19,9% ausentes.

### Resumo do Bloco A — o que levar para os próximos passos

| Achado | Onde será tratado |
|---|---|
| Ausentes: `Cabin` 77,1%, `Age` 19,9%, `Embarked` 2 linhas | Passo 7 (valores ausentes) |
| `Fare` é **por bilhete**, com forte assimetria à direita; 15 tarifas zero são cortesias, não erro | Passo 8 (*outliers*): avaliar `Fare` por pessoa |
| `SibSp`/`Parch` têm máximos altos (8 e 6), mas são famílias reais | Passo 8 (*outliers*) |
| `PassengerId`, `Name`, `Ticket`, `Cabin` são identificadores ou de alta cardinalidade | Passo 8 (redução de colunas), com possível extração de título e deck |
| `Sex` e `Embarked` são nominais; `Pclass` é ordinal | Passo 9 (codificação) |
| `Age` e `Fare` são contínuas, em escalas muito diferentes | Passo 10 (normalização/padronização) |
| Alvo desbalanceado (38,4% sobreviventes) | Passos 4 e 6 |
| Nenhuma linha duplicada | Nenhuma ação necessária |